# Lesson 03 - Agentic Design Patterns

이 수업에서는 효과적인 AI 에이전트를 구축하기 위한 세 가지 기본 설계 패턴을 살펴봅니다:

1. **명확한 에이전트 지침** — 에이전트의 행동을 안내하는 정확하고 역할을 정의하는 프롬프트 작성
2. **Pydantic 모델을 사용한 구조화된 출력** — 에이전트가 예측 가능하고 검증된 데이터를 반환하도록 보장
3. **단일 책임 에이전트** — 각각 하나의 작업을 잘 수행하는 집중된 에이전트 설계

각 패턴을 **여행지 추천자** 시나리오에 적용하여 목적지 추천, 가용성 확인, 물류 처리 등을 점진적으로 구축하는 시스템을 만들어 나갈 것입니다.


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 패턴 1: 명확한 에이전트 지침

가장 효과적인 패턴은 가장 간단한 것: 에이전트를 위한 명확하고 상세한 지침 작성입니다.

좋은 지침은 다음을 정의합니다:
- **누가** 에이전트인지 (페르소나와 어조)
- **무엇을** 해야 하는지 (단계별 책임)
- **어떻게** 행동해야 하는지 (제한 조건과 스타일)

아래에서는 모든 응답을 형성하는 명시적인 지침과 함께 여행 컨시어지 에이전트를 만듭니다.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## 패턴 2: Pydantic 모델을 사용한 구조화된 출력

자유 형식 텍스트는 대화에 유용하지만 하위 시스템은 구조화된 데이터가 필요합니다.  
**Pydantic 모델**과 **도구 함수**를 함께 사용하여 다음을 할 수 있습니다:

- 에이전트 출력에 대한 정확한 스키마 정의
- 응답 자동 검증
- 에이전트 결과를 애플리케이션 로직에 안정적으로 통합

또한 에이전트가 실제 데이터에 근거하여 추천할 수 있도록 목적지 세부 정보를 반환하는 도구도 소개합니다.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: 단일 책임 에이전트

복잡한 작업은 각기 단일 책임을 가진 여러 집중된 에이전트로 작업을 분할하는 것이 유리합니다:

- 장소와 가용성을 아는 **목적지 전문가**
- 항공편, 호텔 및 일정 관리를 담당하는 **물류 기획자**

이는 소프트웨어 공학 원칙인 *관심사의 분리*를 반영합니다 — 각 에이전트는 독립적으로 테스트, 유지 관리 및 개선이 더 쉽습니다.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## 요약

이 수업에서는 여행 추천 시나리오에 세 가지 에이전트 설계 패턴을 적용했습니다:

| 패턴 | 핵심 아이디어 | 이점 |
|---|---|---|
| **명확한 지침** | 페르소나, 책임, 제약 조건을 사전에 정의 | 일관되고 브랜드에 맞는 에이전트 행동 |
| **구조화된 출력** | 응답 형식으로 Pydantic 모델 사용 | 검증되고 기계가 읽을 수 있는 결과 |
| **단일 책임 원칙** | 각 에이전트에 하나의 집중된 업무 부여 | 테스트, 유지 관리 및 조합 용이 |

이 패턴들은 자연스럽게 조합됩니다 — 명확한 지침과 구조화된 출력을 단일 책임 에이전트 안에 결합하여 견고하고 프로덕션 준비가 된 시스템을 구축할 수 있습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 최선을 다하고 있으나, 자동 번역에는 오류나 부정확한 내용이 포함될 수 있습니다. 원본 문서는 해당 언어의 원문이 권위 있는 출처로 간주되어야 합니다. 중요한 정보의 경우, 전문적인 인간 번역을 권장합니다. 본 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
